In [2]:
import os
import re
import yaml
import torch
from transformers import AutoTokenizer
from model.sar_vlm import SARVLM, build_sar_encoder
from dataset import SARVLMDataset


def load_config(config_path):
    with open(config_path, "r") as f:
        return yaml.safe_load(f)


def find_latest_checkpoint(checkpoint_root):
    candidates = []

    for name in os.listdir(checkpoint_root):
        match = re.match(r"step_(\d+)$", name)

        if match:
            step = int(match.group(1))
            path = os.path.join(checkpoint_root, name)

            if os.path.isdir(path):
                candidates.append((step, path))

    if not candidates:
        raise RuntimeError(
            f"No step_* checkpoints found in {checkpoint_root}"
        )

    candidates.sort(key=lambda x: x[0])

    return candidates[-1]

I0000 00:00:1787763743.207136  397081 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
import gc

# Clear GPU cache before starting
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

In [ ]:
import gc

# ---------------------------------------------------------
# Cleanup from previous notebook runs
# ---------------------------------------------------------

if "vlm" in globals():
    del vlm

if "sar_encoder" in globals():
    del sar_encoder

if "val_dataset" in globals():
    del val_dataset

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()


# ---------------------------------------------------------
# Config
# ---------------------------------------------------------

config = load_config("train_config.yaml")

c_data = config["data"]
c_model = config["model"]
c_lora = config["lora"]
c_train = config["training"]

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# ---------------------------------------------------------
# Find latest checkpoint
# ---------------------------------------------------------

checkpoint_root = "./checkpoints"

step, checkpoint_dir = find_latest_checkpoint(
    checkpoint_root
)

print("=" * 70)
print(f"Latest checkpoint: step_{step}")
print(f"Path: {checkpoint_dir}")
print("=" * 70)

projector_path = os.path.join(
    checkpoint_dir,
    "projector.pth"
)

if not os.path.exists(projector_path):
    raise FileNotFoundError(projector_path)


# ---------------------------------------------------------
# Tokenizer
# ---------------------------------------------------------

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    c_model["vicuna_path"],
    use_fast=False
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.unk_token


# ---------------------------------------------------------
# SAR encoder
# ---------------------------------------------------------

print("Loading SAR encoder...")

sar_encoder = build_sar_encoder(
    checkpoint_path=c_model["encoder_checkpoint"],
    freeze=True,
    d_sar=c_model["d_sar"]
)


# ---------------------------------------------------------
# Build full VLM
# ---------------------------------------------------------

print("Loading Vicuna + LoRA architecture...")

vlm = SARVLM.from_vicuna(
    vicuna_path=c_model["vicuna_path"],
    sar_encoder=sar_encoder,
    d_sar=c_model["d_sar"],
    n_visual=c_model["n_visual"],
    lora_r=c_lora["r"],
    lora_alpha=c_lora["alpha"],
    lora_dropout=c_lora["dropout"],
    lora_target_modules=c_lora["target_modules"],
    apply_lora=True,
    torch_dtype=torch.float32
)


# ---------------------------------------------------------
# Load LoRA checkpoint
# ---------------------------------------------------------

print("Loading LoRA checkpoint...")

# from_vicuna(..., apply_lora=True) has already created
# the PEFT model. Load the saved adapter weights directly.
vlm.hybrid_vicuna.load_adapter(
    checkpoint_dir,
    adapter_name="default"
)


# ---------------------------------------------------------
# Load SAR -> Vicuna projector
# ---------------------------------------------------------

print("Loading projector...")

projector_state = torch.load(
    projector_path,
    map_location="cpu",
    weights_only=True
)

vlm.projector.load_state_dict(
    projector_state,
    strict=True
)


# ---------------------------------------------------------
# Move model to GPU
# ---------------------------------------------------------

print("Moving model to device...")

vlm = vlm.to(device)
vlm.eval()

print("Model loaded successfully.")
print(f"Using checkpoint: step_{step}")


# ---------------------------------------------------------
# Load validation dataset
# ---------------------------------------------------------

print("Loading validation dataset...")

val_dataset = SARVLMDataset(
    c_data["val_jsonl"],
    c_data["data_root"],
    tokenizer,
    max_length=c_train["max_length"]
)

print(f"Validation dataset size: {len(val_dataset)}")

print("=" * 70)

Latest checkpoint: step_4000
Path: ./checkpoints/step_4000
Loading tokenizer...
Loading SAR encoder...
check point path is  /home/saishruti/Research1/Shreyank_20_credit/MyModels/SarEncPlusVicuna/mars_base_sar_encoder_only.pth
Loading Vicuna + LoRA architecture...
[SARVLM] Loading Vicuna config from lmsys/vicuna-7b-v1.5 ...
[SARVLM] Loading Vicuna weights from lmsys/vicuna-7b-v1.5 ...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
    sample = val_dataset[0]

    sar_input = sample["sar_input"].unsqueeze(0).to(
        device,
        dtype=torch.float32
    )
    # sar_input = torch.zeros_like(
    #     sample["sar_input"]
    # ).unsqueeze(0).to(
    #     device,
    #     dtype=torch.float32
    # )
    input_ids = sample["input_ids"]
    labels = sample["labels"]

    # ---------------------------------------------------------
    # Reconstruct prompt
    # ---------------------------------------------------------

    prompt_mask = labels == -100

    prompt_ids = input_ids[
        prompt_mask
    ].unsqueeze(0).to(device)

    prompt_attention_mask = torch.ones_like(
        prompt_ids
    ).to(device)

    print("\nPrompt:")
    print(
        tokenizer.decode(
            prompt_ids[0],
            skip_special_tokens=False
        )
    )
    target_ids = sample["labels"][sample["labels"] != -100]

    print("GROUND TRUTH:")
    print(
        repr(
            tokenizer.decode(
                target_ids,
                skip_special_tokens=False
            )
        )
    )
    # ---------------------------------------------------------
    # Generate
    # ---------------------------------------------------------

    print("\nGenerating...")
    
    with torch.no_grad():

        output_ids = vlm.generate(
            sar_input=sar_input,
            input_ids=prompt_ids,
            attention_mask=prompt_attention_mask,
            max_new_tokens=50,
            do_sample=False
        )

    # ---------------------------------------------------------
    # Decode
    # ---------------------------------------------------------

    generated_ids = output_ids[0]

    print("\nRaw generated token IDs:")
    print(generated_ids.tolist())

    print("\nRaw decoded output:")
    print(
        repr(
            tokenizer.decode(
                generated_ids,
                skip_special_tokens=False
            )
        )
    )
    print("INPUT IDS:")
    print(sample["input_ids"].tolist())

    print("\nLABELS:")
    print(sample["labels"].tolist())

    valid_labels = sample["labels"][
        sample["labels"] != -100
    ]

    print("\nVALID LABEL IDS:")
    print(valid_labels.tolist())

    print("\nVALID LABEL TEXT:")
    print(
    repr(
        tokenizer.decode(
                valid_labels,
                skip_special_tokens=False
            )
        )
    )
    generated_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )
    print("Prompt shape:", prompt_ids.shape)
    print("Output shape:", output_ids.shape)

    print("Prompt length:", prompt_ids.shape[1])
    print("Output length:", output_ids.shape[1])

    print("Raw output IDs:")
    print(output_ids[0].tolist())

    print("\n" + "=" * 70)
    print("GENERATED ANSWER:")
    print(repr(generated_text))
    print("=" * 70)

In [ ]:
# =========================================================
# TEACHER-FORCED LOGIT INSPECTION
# =========================================================
from transformers.masking_utils import create_causal_mask
from dataset import collate_fn
def make_sar_visual_mask(num_visual_tokens: int):
    """
    Returns a mask function compatible with create_causal_mask(or_mask_function=...).

    The function returns True (attention allowed) when both the query token
    and the key/value token are within the SAR visual prefix.

    OR-ing with the causal mask produces the desired pattern:

        SAR → SAR  : 1  (bidirectional, added by this function)
        SAR → text : 0  (q_idx < N_v but kv_idx >= N_v → function returns False,
                         causal mask also False since SAR precedes text → 0)
        text→ SAR  : 1  (q_idx >= N_v, kv_idx < N_v → function False, but
                         causal mask True because kv_idx <= q_idx → 1)
        text→ text : causal lower-triangular

    Args:
        num_visual_tokens: number of leading SAR visual positions (N_visual).

    Returns:
        Callable with signature (batch_idx, head_idx, q_idx, kv_idx) -> bool.
    """
    def visual_mask(
        batch_idx: int,
        head_idx: int,
        q_idx: int,
        kv_idx: int,
    ) -> bool:
        # Both positions must be inside the visual prefix.
        return (q_idx < num_visual_tokens) & (kv_idx < num_visual_tokens)

    return visual_mask
sample = val_dataset[0]

batch = collate_fn(
    [sample],
    tokenizer
)

sar_input = batch["sar_input"].to(
    device,
    dtype=torch.float32
)

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)
vlm.eval()

print("attention mask \n", attention_mask)

with torch.no_grad():

    outputs = vlm(
        sar_input=sar_input,
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

    logits = outputs.logits

print("\n" + "=" * 70)
print("TEACHER-FORCED LOGIT INSPECTION")
print("=" * 70)

print("Model loss:", outputs.loss.item())
print("Logits shape:", logits.shape)

# ---------------------------------------------------------
# Find first answer token
# ---------------------------------------------------------

# labels == -100 means prompt.
# First non--100 position is the first answer token.

answer_positions = torch.where(
    labels[0] != -100
)[0]

first_answer_position = answer_positions[0].item()

print(
    "First answer position in input:",
    first_answer_position
)

# ---------------------------------------------------------
# IMPORTANT:
# Causal LM predicts token at position t+1 using
# logits from position t.
#
# Therefore to predict the first answer token,
# we use logits from position immediately BEFORE it.
# ---------------------------------------------------------

prediction_position = first_answer_position - 1

first_token_logits = logits[
    0,
    prediction_position
]

first_token_probs = torch.softmax(
    first_token_logits,
    dim=-1
)

# ---------------------------------------------------------
# Ground truth first token
# ---------------------------------------------------------

ground_truth_token_id = labels[
    0,
    first_answer_position
].item()

ground_truth_token = tokenizer.decode(
    [ground_truth_token_id],
    skip_special_tokens=False
)

ground_truth_probability = first_token_probs[
    ground_truth_token_id
].item()

print("\nGROUND TRUTH FIRST TOKEN")
print("-----------------------")
print("Token ID:", ground_truth_token_id)
print("Token:", repr(ground_truth_token))
print(
    "Probability:",
    f"{ground_truth_probability:.8f}"
)

# ---------------------------------------------------------
# What token does the model predict?
# ---------------------------------------------------------

predicted_token_id = torch.argmax(
    first_token_probs
).item()

predicted_token = tokenizer.decode(
    [predicted_token_id],
    skip_special_tokens=False
)

predicted_probability = first_token_probs[
    predicted_token_id
].item()

print("\nMODEL TOP-1 PREDICTION")
print("----------------------")
print("Token ID:", predicted_token_id)
print("Token:", repr(predicted_token))
print(
    "Probability:",
    f"{predicted_probability:.8f}"
)

# ---------------------------------------------------------
# Top 20 predictions
# ---------------------------------------------------------

top_probs, top_ids = torch.topk(
    first_token_probs,
    k=20
)

print("\nTOP 20 PREDICTIONS")
print("------------------")

for rank, (prob, token_id) in enumerate(
    zip(top_probs.tolist(), top_ids.tolist()),
    start=1
):

    token = tokenizer.decode(
        [token_id],
        skip_special_tokens=False
    )

    print(
        f"{rank:2d}. "
        f"ID={token_id:6d} "
        f"P={prob:.8f} "
        f"token={repr(token)}"
    )

# ---------------------------------------------------------
# Print probability of "The"
# ---------------------------------------------------------

the_ids = tokenizer.encode(
    "The",
    add_special_tokens=False
)

if len(the_ids) == 1:

    the_id = the_ids[0]

    print("\n'The' TOKEN")
    print("-----------")
    print("Token ID:", the_id)
    print(
        "Probability:",
        f"{first_token_probs[the_id].item():.8f}"
    )

print("=" * 70)

In [ ]:
vlm.eval()

sample = val_dataset[0]

# Use EXACTLY the same collate function as validation
batch = collate_fn([sample], tokenizer)

sar_input = batch["sar_input"].to(
    device,
    dtype=torch.float32
)

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)

print("Input shape:", input_ids.shape)
print("SAR shape:", sar_input.shape)
print("Attention mask:", attention_mask)

with torch.no_grad():

    with torch.cuda.amp.autocast(
        enabled=torch.cuda.is_available()
    ):

        outputs = vlm(
            sar_input=sar_input,
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

print("\n" + "=" * 70)
print("SINGLE SAMPLE VALIDATION LOSS")
print("=" * 70)

print("Loss:", outputs.loss.item())
print("Logits shape:", outputs.logits.shape)

print("=" * 70)